# Sudz: train your voice

This notebook teaches a voice model your singing, so Sudz can re-sing any other vocal in your voice (**Pitch › Sing it in my voice** in Sudz).

**You need:** 15–30 minutes of your own dry, solo singing (WAV or AIFF), and a free Google account.
**It takes:** about 1–2 hours on Colab's free GPU. Your progress is saved to your Google Drive, so if Colab disconnects you can run the cells again and it carries on.

**Before you start:** *Runtime › Change runtime type › T4 GPU*, then run each cell in order (the ▶ button on its left).

Only train on your own voice, or on a singer who has given you permission. Uses [Applio](https://github.com/IAHispano/Applio) 3.7.1 (MIT licence).

In [ ]:
# @title 1. Check the GPU
import shutil, subprocess
msg = "No GPU yet. At the top: Runtime › Change runtime type (or the ▾ next to Connect) › choose T4 GPU › Save, then run this cell again."
if shutil.which("nvidia-smi") is None:
    raise SystemExit(msg)
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip():
    raise SystemExit(msg)
print("GPU:", r.stdout.strip(), "- good, carry on with cell 2.")


In [ ]:
# @title 2. Connect Google Drive (your vocals go in, your voice model comes out)
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
HOME = Path("/content/drive/MyDrive/Sudz Voice")
(HOME / "my vocals").mkdir(parents=True, exist_ok=True)
(HOME / "export").mkdir(parents=True, exist_ok=True)
(HOME / "training").mkdir(parents=True, exist_ok=True)
print("Put your vocal files in:  My Drive › Sudz Voice › my vocals")
print("Files there now:", sorted(p.name for p in (HOME / "my vocals").iterdir()) or "none yet")

In [ ]:
# @title 3. Install the training software (about 5 minutes)
%cd /content
!git config --global advice.detachedHead false
!rm -rf /content/Applio && git clone -q --depth 1 --branch 3.7.1 https://github.com/IAHispano/Applio
%cd /content/Applio
!apt -qq update -y > /dev/null && apt -qq install -y portaudio19-dev > /dev/null
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null
!/root/.local/bin/uv venv .venv --python 3.12 --seed -q
!/root/.local/bin/uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --python /content/Applio/.venv/bin/python
!/content/Applio/.venv/bin/python core.py prerequisites --models --pretraineds-hifigan --exe
!cp assets/config_template.json assets/config.json
!curl -fsSL -o /content/sudz_export.py https://raw.githubusercontent.com/davidsudwarts-hub/sudz/main/voice/sudz_export.py
# keep training progress on Drive, so a disconnect doesn't lose it
!rm -rf /content/Applio/logs && ln -s "/content/drive/MyDrive/Sudz Voice/training" /content/Applio/logs
import os
if not os.path.exists("/content/Applio/.venv/bin/python"):
    raise SystemExit("The install didn't finish: scroll up for the first red error and send it to Claude.")
print("Installed. Now run cell 4.")

In [ ]:
# @title 4. Your vocals
# @markdown Name for your voice (letters and numbers only):
voice_name = "MyVoice"  # @param {type:"string"}
# @markdown If "my vocals" on Drive is empty, tick this to upload files from your computer instead:
upload_now = False  # @param {type:"boolean"}
import re, soundfile as sf
voice_name = re.sub(r"[^A-Za-z0-9_]", "", voice_name) or "MyVoice"
vocals = HOME / "my vocals"
if upload_now:
    from google.colab import files
    for name, data in files.upload().items():
        (vocals / name).write_bytes(data)
total = 0.0
for p in sorted(vocals.iterdir()):
    try:
        i = sf.info(str(p)); total += i.duration; print(f"  {p.name}: {i.duration/60:.1f} min")
    except Exception:
        print(f"  {p.name}: not an audio file Sudz can use (skipped by training)")
print(f"Total: {total/60:.1f} minutes of singing")
if total < 8 * 60:
    print("That's on the short side: 15–30 minutes gives a much better voice. It will still train.")

In [ ]:
# @title 5. Prepare the recordings (cut into pieces, pitch and features), about 5–10 minutes
# --- checks: this needs cells 3 (install) and 4 (your vocals) to have run in this session
import os
if not os.path.exists("/content/Applio/.venv/bin/python"):
    raise SystemExit("The training software isn't installed in this session. Run cells 1, 2, 3 and 4 again in order (Colab forgets them when it restarts), then this one.")
if "voice_name" not in globals() or "vocals" not in globals():
    raise SystemExit("Run cell 4 (your vocals) first, then this one.")
from multiprocessing import cpu_count
cores = cpu_count()
%cd /content/Applio
!/content/Applio/.venv/bin/python core.py preprocess --model-name "{voice_name}" --dataset-path "{vocals}" --sample-rate 40000 --cpu-cores {cores} --cut-preprocess Automatic --chunk-len 3.0 --overlap-len 0.3 --normalization-mode none
!/content/Applio/.venv/bin/python core.py extract --model-name "{voice_name}" --f0-method rmvpe --sample-rate 40000 --cpu-cores {cores} --gpu 0 --embedder-model contentvec --include-mutes 2

In [ ]:
# @title 6. Train (about 1–2 hours). If Colab disconnects, run cells 1–4 again and then this one: it carries on.
# @markdown 300 rounds suits 15–30 minutes of singing. More isn't always better.
# --- checks: this needs cells 3 (install) and 4 (your vocals) to have run in this session
import os
if not os.path.exists("/content/Applio/.venv/bin/python"):
    raise SystemExit("The training software isn't installed in this session. Run cells 1, 2, 3 and 4 again in order (Colab forgets them when it restarts), then this one.")
if "voice_name" not in globals() or "vocals" not in globals():
    raise SystemExit("Run cell 4 (your vocals) first, then this one.")
from multiprocessing import cpu_count
cores = cpu_count()
total_epoch = 300  # @param {type:"integer"}
%cd /content/Applio
!/content/Applio/.venv/bin/python core.py train --model-name "{voice_name}" --save-every-epoch 25 --save-only-latest --total-epoch {total_epoch} --sample-rate 40000 --batch-size 8 --gpu 0 --pretrained --vocoder HiFi-GAN

In [ ]:
# @title 7. Make the files for Sudz and save them to Drive
# --- checks: this needs cells 3 (install) and 4 (your vocals) to have run in this session
import os
if not os.path.exists("/content/Applio/.venv/bin/python"):
    raise SystemExit("The training software isn't installed in this session. Run cells 1, 2, 3 and 4 again in order (Colab forgets them when it restarts), then this one.")
if "voice_name" not in globals() or "vocals" not in globals():
    raise SystemExit("Run cell 4 (your vocals) first, then this one.")
from multiprocessing import cpu_count
cores = cpu_count()
import glob, json, os, shutil, numpy as np
%cd /content/Applio
model_dir = f"/content/Applio/logs/{voice_name}"
cands = sorted(glob.glob(f"{model_dir}/{voice_name}_*e_*s.pth") + glob.glob(f"{model_dir}/{voice_name}.pth"), key=os.path.getmtime)
if not cands:
    raise SystemExit("No trained model found yet: run cell 6 first.")
pth = cands[-1]; print("Using", os.path.basename(pth))
out = f"/content/{voice_name}.onnx"
!/content/Applio/.venv/bin/python /content/sudz_export.py voice "{pth}" "{out}"
# your typical pitch, so Sudz can move other vocals into your range
f0 = np.concatenate([np.load(f) for f in glob.glob(f"{model_dir}/f0_voiced/*.npy")])
meta = json.load(open(out + ".json")); meta["name"] = voice_name; meta["median_hz"] = float(np.median(f0[f0 > 0]))
json.dump(meta, open(out + ".json", "w"), indent=1); print(meta)
content = "/content/sudz_content.onnx"
if not os.path.exists(HOME / "export" / "sudz_content.onnx"):
    !/content/Applio/.venv/bin/python /content/sudz_export.py content /content/Applio/rvc/models/embedders/contentvec "{content}"
    shutil.copy(content, HOME / "export" / "sudz_content.onnx")
shutil.copy(out, HOME / "export" / os.path.basename(out)); shutil.copy(out + ".json", HOME / "export" / (os.path.basename(out) + ".json"))
print()
print("Done. In Google Drive › Sudz Voice › export you now have:")
for p in sorted((HOME / "export").iterdir()): print(f"  {p.name}  ({p.stat().st_size/1e6:.0f} MB)")
print("Download them to your Mac, then in Sudz: Pitch › Sing it in my voice › Add a voice model (both the .onnx and the .onnx.json), and choose sudz_content.onnx once.")

In [ ]:
# @title 8. (Optional) Hear it: convert a short clip here to check the voice before downloading
# @markdown Upload a few seconds of someone else's dry vocal when asked.
from google.colab import files
from IPython.display import Audio, display
up = files.upload()
if up:
    name = list(up)[0]; open(f"/content/test_in_{name}", "wb").write(up[name])
    %cd /content/Applio
    !/content/Applio/.venv/bin/python core.py infer --pitch 0 --index-rate 0 --protect 0.33 --f0-method rmvpe --input-path "/content/test_in_{name}" --output-path /content/test_out.wav --pth-path "{pth}" --index-path "" --export-format WAV --embedder-model contentvec
    display(Audio("/content/test_out.wav"))